# NAVSIM long-episode dataset statistics

Inline version of the first three report pages of `exp_navsim/visualize_dataset.py`
(episode lengths, driven distance, steering angle), plus a shortlist of episodes
worth watching with `tools/remote_viewer/view_episode.py`.

Run on the cluster in the `navsim` env. The scan below is the only slow cell;
everything after it re-runs instantly.

In [ ]:
%cd /scratch/local/velikanov/work/orbis
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from tqdm.auto import tqdm

from exp_navsim.data.navsim_base import NavsimLongBase
# Reuse the report's own statistic and thresholds, so the numbers here and in
# navsim_episodes.pdf cannot drift apart.
from exp_navsim.visualize_dataset import (
    trajectory_stats, MIN_DISPLACEMENT, HIST_MIN_DISTANCE, HIST_MIN_ANGLE,
)

CONFIG = "exp_navsim/config.yaml"
cfg = OmegaConf.load(CONFIG)
ds = NavsimLongBase.from_config(cfg)
print(f"dataset={cfg.dataset}  episodes={len(ds)}  frame_rate={ds.frame_rate} Hz")

In [ ]:
# One pass over every episode. Image-free: episode_trajectory only parses the pose
# metadata out of the pickles, so this touches no camera blobs.
lengths, distances, steerings, displacements = [], [], [], []
for i in tqdm(range(len(ds))):
    traj = ds.episode_trajectory(i)
    lengths.append(len(traj))
    dist, steer, disp = trajectory_stats(traj)
    distances.append(dist); steerings.append(steer); displacements.append(disp)

lengths = np.asarray(lengths)
distances = np.asarray(distances)
steerings = np.asarray(steerings)
displacements = np.asarray(displacements)

# Below MIN_DISPLACEMENT the start->end vector is too short for its angle to carry
# any signal, so those episodes are dropped from every steering plot.
moved = displacements >= MIN_DISPLACEMENT
print(f"{moved.sum()}/{len(ds)} episodes moved at least {MIN_DISPLACEMENT:g} m")

## Episode lengths

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lengths, bins=min(30, max(len(set(lengths.tolist())), 1)), color="tab:blue", alpha=0.8)
ax.set_xlabel("episode length (frames)"); ax.set_ylabel("count")
ax.set_title("Distribution of episode lengths"); ax.grid(True, alpha=0.3)
plt.show()

print(f"length (frames)  min={lengths.min()}  max={lengths.max()}  "
      f"mean={lengths.mean():.1f}  median={np.median(lengths):.0f}")
print(f"at {ds.frame_rate} Hz that is {lengths.min() / ds.frame_rate:.0f}"
      f"..{lengths.max() / ds.frame_rate:.0f} s of video per episode")

## Ego motion

In [ ]:
def motion_hists(distances, steerings, title):
    """Distance + steering histograms side by side (report pages 2 and 3)."""
    fig, (ax_d, ax_s) = plt.subplots(1, 2, figsize=(12, 4.5))
    fig.suptitle(title, fontsize=14)
    for ax, values, color, xlabel in (
        (ax_d, distances, "tab:blue", "distance driven (m)"),
        (ax_s, steerings, "tab:orange", "steering angle (deg)"),
    ):
        if len(values):
            ax.hist(values, bins=30, color=color, alpha=0.8)
            ax.set_title(f"median={np.median(values):.1f}   n={len(values)}")
        else:
            ax.text(0.5, 0.5, "no episodes", ha="center", va="center", transform=ax.transAxes)
        ax.set_xlabel(xlabel); ax.set_ylabel("count"); ax.grid(True, alpha=0.3)
    fig.tight_layout(); plt.show()


motion_hists(distances, steerings[moved], "Ego motion - all episodes")

In [ ]:
# The full-range histograms are dominated by short, straight episodes; zoom on the
# tails to see the ones actually worth visualizing.
motion_hists(distances[distances >= HIST_MIN_DISTANCE],
             steerings[moved][np.abs(steerings[moved]) >= HIST_MIN_ANGLE],
             f"Ego motion - distance >= {HIST_MIN_DISTANCE:g} m / "
             f"|steering| >= {HIST_MIN_ANGLE:g} deg")

## Which episodes to watch

In [ ]:
# Joint view of both criteria: the episodes worth watching sit in the upper right
# quadrant marked by the two thresholds.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(distances[moved], np.abs(steerings[moved]), s=8, alpha=0.3)
ax.axvline(HIST_MIN_DISTANCE, color="tab:red", ls="--", lw=1)
ax.axhline(HIST_MIN_ANGLE, color="tab:red", ls="--", lw=1)
ax.set_xlabel("distance driven (m)"); ax.set_ylabel("|steering angle| (deg)")
ax.set_title("Distance vs turn, per episode"); ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Shortlist: the longest sharply-turning episodes. Feed an index straight to the
# viewer on the local PC:
#     python view_episode.py --token <token> --episode <idx>
interesting = np.flatnonzero(moved & (distances >= HIST_MIN_DISTANCE)
                             & (np.abs(steerings) >= HIST_MIN_ANGLE))
order = interesting[np.argsort(-lengths[interesting])][:15]
print(f"{len(interesting)} episodes pass both thresholds; longest 15:\n")
print(f"{'idx':>6} {'frames':>7} {'dist m':>8} {'steer deg':>10}  token")
for i in order:
    print(f"{i:>6} {lengths[i]:>7} {distances[i]:>8.1f} {steerings[i]:>10.1f}  "
          f"{ds.episode_token(i)}")